# ZizAIの初回セットアップ（Windows / VS Code）

**最初に選ぶカーネルは、このプロジェクトの `.venv\Scripts\python.exe` です。** 新規の場合は、Notebookのコードセルを実行する前に作成します。

1. VS Codeで `pyproject.toml` があるフォルダーを開きます。Python拡張とJupyter拡張が必要です。
2. VS Codeの **ターミナル → 新しいターミナル** でPowerShellを開き、そのフォルダーで次の2行を実行します。`py -3.14` はインストール済みのPython 3.14を使う指定です。

```powershell
py -3.14 -m venv .venv
& ".\.venv\Scripts\python.exe" -m pip install ipykernel
```

すでにプロジェクトの `.venv` がある場合は、1行目は省略します。`py -3.14` が見つからなければ `py -0p` で確認し、手元のPython 3.12以降を指定してください。

3. このNotebookの右上で **カーネルの選択 → 他のカーネルを選択 → Python環境** を開き、`.venv\Scripts\python.exe` を選びます。[VS Codeのカーネル選択手順](https://code.visualstudio.com/docs/datascience/jupyter-kernel-management)
4. 最初のコードセル **0** の左にある **▶** を押します。**0 → ① → ② → ③** と1セルずつ実行し、③の後で同じカーネルを再起動します。④は再起動後に実行します。

macOS・Linuxでは、作成に `python3 -m venv .venv`（Python 3.12以降）、`ipykernel` の導入に `.venv/bin/python -m pip install ipykernel` を使い、カーネルは `.venv/bin/python` を選びます。

## 0. セットアップの設定を確認する

初回はこのコードセルをそのまま実行してください。使うコネクタは①の後にチェックボックスで選びます。別のコネクタ共通フォルダーを使う場合は `shared_connectors` で指定します。

導入に成功したら、ZizAIルート直下のビルド作業フォルダーを②で片付けます。保持したい場合は `clean_build = False` にします。

すでにuvを導入済みで、ZizAIとライブラリのインストールに使う場合は `installer = "uv"` にします。既存の別環境を利用する場合は、その環境に `ipykernel` があることを確認してNotebook右上で選び、`use_project_venv = False` にします。②と③の対象は手順0に表示されます.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import sysconfig
import tomllib

project_root = None              # 自動判定できないときは例: "C:/tools/ZizAI"
shared_connectors = None         # 省略時は ZizAI/connectors
use_project_venv = True
installer = "pip"                # または "uv"
clean_build = True               # ②でビルド作業フォルダーを片付ける

selected_connectors = None
installed_connectors = None

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以降のカーネルを選択してください。")

start = Path(project_root or Path.cwd()).expanduser().resolve()
roots = (start,) if project_root else (start, *start.parents)
root = next((p for p in roots if (p / "pyproject.toml").is_file()
             and (p / "src/forms.py").is_file() and (p / "connectors").is_dir()), None)
if root is None:
    raise ValueError("ZizAIのルートを開くか、project_rootにルートのパスを指定してください。")

catalog = Path(shared_connectors or root / "connectors").expanduser().resolve()
available_connectors = sorted({definition.parent.name
                               for definition in catalog.glob("*/*_definition.yaml")
                               if not definition.parent.name.startswith(".")
                               and definition.parent.name != "__pycache__"})
if not available_connectors:
    raise ValueError("コネクタYAMLがある共通フォルダーを指定してください。")

project = tomllib.loads((root / "pyproject.toml").read_text(encoding="utf-8"))["project"]
if installer not in {"pip", "uv"}:
    raise ValueError('installerは "pip" または "uv" にしてください。')

environment = root / ".venv" if use_project_venv else Path(sys.prefix)
python = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if use_project_venv:
    if environment.is_symlink() or environment.is_junction():
        raise ValueError(".venvがリンクです。既存環境として利用する場合は、対象カーネルを選びuse_project_venv = Falseにしてください。")
    if not (environment / "pyvenv.cfg").is_file() or not python.is_file():
        raise RuntimeError("先に冒頭のターミナル手順で.venvを作ってください。")
    if Path(sys.prefix).resolve() != environment.resolve():
        raise RuntimeError(f"Notebook右上のカーネル選択で、このPythonを選んでください: {python}")
else:
    python = Path(sys.executable)

def run_target(*args, **kwargs):
    return subprocess.run([str(python), "-I", *args], check=True, **kwargs)

def install_for_target(*requirements):
    if installer == "uv":
        uv = shutil.which("uv")
        if uv is None:
            raise RuntimeError("uvが見つかりません。セル0の設定を確認してください。")
        subprocess.run([uv, "pip", "install", "--python", str(python), *requirements], check=True)
    else:
        run_target("-m", "pip", "--isolated", "install", *requirements)


print(f"ZizAI: {root}")
print(f"導入先のPython: {python}")
print(f"共通コネクタ: {catalog}")
print(f"インストーラー: {installer}")

## ① 選択したPythonを確認する

このセルでPythonのバージョンとインストール先を確認します。`準備済み` と表示されたら、下のコネクタ選択を実行してください。

In [ ]:
if installer == "uv" and not shutil.which("uv"):
    raise RuntimeError('uvが見つかりません。先に導入するか、0で installer = "pip" にしてください。')

run_target("-c", "from pathlib import Path; import sys; "
           "assert sys.version_info >= (3, 12), 'Python 3.12以上が必要です。'; "
           "assert Path(sys.prefix).resolve() == Path(sys.argv[1]).resolve(), sys.prefix",
           str(environment))
print(f"準備済み: {python}")

## 使うコネクタを選ぶ

下のコードセルを実行し、使うコネクタを1つ以上選びます。チェックボックスが表示されたらチェックを入れます。入力が求められたら `csv,excel` のようにカンマで区切って入力します。初回はCSV、再実行時は登録済みの選択が初期値です。選択後、②へ進みます。

In [ ]:
selected_connectors = None
installed_connectors = None
settings_path = Path(sysconfig.get_path("purelib")) / "zizai-connectors.json"
settings = json.loads(settings_path.read_text(encoding="utf-8")) if settings_path.is_file() else {}
saved = settings.get("enabled_connectors") if settings.get("connectors_directory") == str(catalog) else None
initial = [name for name in available_connectors if name in (saved or [])]
if not initial:
    initial = ["csv"] if "csv" in available_connectors else available_connectors[:1]

try:
    from ipywidgets import Checkbox, VBox
except ImportError:
    print(f"選べるコネクタ: {', '.join(available_connectors)}")
    entered = input(f"使うコネクタをカンマ区切りで入力 [{', '.join(initial)}]: ").strip()
    chosen = tuple(dict.fromkeys(name.strip() for name in entered.split(",") if name.strip()))
    if not entered:
        chosen = tuple(initial)
    unknown = set(chosen) - set(available_connectors)
    if unknown:
        raise ValueError(f"一覧にないコネクタ: {', '.join(sorted(unknown))}")

    def selected_connectors():
        return chosen
else:
    from IPython.display import display

    connector_checkboxes = {name: Checkbox(description=name, value=name in initial, indent=False)
                            for name in available_connectors}
    display(VBox(list(connector_checkboxes.values())))

    def selected_connectors():
        return tuple(name for name, checkbox in connector_checkboxes.items() if checkbox.value)

## ② ZizAIとライブラリを導入する

選んだコネクタに必要な追加ライブラリを導入します。通常はパッケージ取得に通信が必要です。Vectorのモデル取得やBigQueryへの接続は、ここでは行いません。

導入成功後、ZizAIルート直下の `build/` がセットアップの作業フォルダーと確認できた場合に片付けます。

In [ ]:
installed_connectors = None
if selected_connectors is None:
    raise RuntimeError("先に使うコネクタを選んでください。")
selected = selected_connectors()
if not selected:
    raise ValueError("使うコネクタに1つ以上チェックを入れてください。")

extras = ["forms", *(name for name in selected if name in project["optional-dependencies"])]
install_for_target("ipykernel", f"{root}[{','.join(extras)}]")
run_target("-c", "import ipykernel, importlib.metadata as m; "
           "print('ZizAI:', m.version('zizai-runtime'))")
installed_connectors = selected
print(f"導入対象: {', '.join(installed_connectors)}")


def is_build_directory(build):
    def plain_directory(folder):
        return folder.is_dir() and not folder.is_symlink() and not folder.is_junction()

    if not plain_directory(build) or build.resolve().parent != root:
        return False
    for child in build.iterdir():
        if not plain_directory(child):
            return False
        if child.name == "lib":
            if any(package.name != "zizai" or not plain_directory(package)
                   for package in child.iterdir()):
                return False
        elif not child.name.startswith("bdist.") or any(child.iterdir()):
            return False
    return not any(entry.is_symlink() or entry.is_junction() for entry in build.rglob("*"))


build = root / "build"
if clean_build and (build.exists() or build.is_symlink() or build.is_junction()):
    if is_build_directory(build):
        try:
            shutil.rmtree(build)
        except OSError as error:
            print(f"buildを片付けられませんでした: {error}")
        else:
            print(f"ビルド作業フォルダーを片付けました: {build}")
    else:
        print(f"作業フォルダーの構成を確認できなかったため保持しました: {build}")

## ③ 選択したコネクタを登録する

②で導入したコネクタを、確認済みのPython環境に登録します。利用するコネクタを変えるときは、選択を変更して②・③を実行します。登録後は同じカーネルを再起動します。

In [ ]:
if installed_connectors is None or selected_connectors() != installed_connectors:
    raise RuntimeError("今の選択で②のライブラリ導入を先に実行してください。")

registration_env = os.environ.copy()
for name in ("ZIZAI_CONFIG_FILE", "ZIZAI_CONNECTORS_DIR"):
    registration_env.pop(name, None)

run_target("-c", "import json, sys; from zizai.connectors import configure_connectors; "
           "print('登録先:', configure_connectors(sys.argv[1], enabled_connectors=json.loads(sys.argv[2])))",
           catalog.as_posix(), json.dumps(installed_connectors), env=registration_env)
print(f"このPythonカーネルを再起動してください: {python}")

## ③の後で、同じカーネルを再起動する

1. Notebook上部の **再起動（Restart）** を押します。右上は、最初に選んだ `.venv` のままです。Pythonは再起動時に③の登録を読み込みます。
2. **0のコードセルをもう一度実行**します。①〜③は実行済みです。
3. ④のコードセルを上から順に実行します。最後の `%zizai_form` でフォームが表示されます。

## ④ マジックを読み込み、フォームを表示する

ここで使うマジックは、通常のNotebookでもカーネルを再起動するたびに読み込みます。フォームで生成したコードは、必要なときに別のセルへコピーして使います。

In [ ]:
if "environment" not in globals():
    raise RuntimeError("カーネル再起動後にセル0を実行してください。")
if Path(sys.prefix).resolve() != environment.resolve():
    raise RuntimeError(f"③で表示されたPythonをカーネルに選び直してください: {python}")

from zizai.core.connector_config import get_connector_directory, get_enabled_connector_folders
from zizai.form_definition import load_definition

if get_connector_directory() != catalog:
    raise RuntimeError("③で登録したコネクタの場所と異なります。登録先とZIZAI_CONFIG_FILE・ZIZAI_CONNECTORS_DIRを確認してください。")

definitions = load_definition()
enabled = get_enabled_connector_folders()
print(f"登録されたコネクタ: {', '.join(enabled or definitions['connectors'])}")
print(f"使用中のPython: {sys.executable}")
print("コネクタ登録を確認しました。下の2セルを実行してください。")

In [ ]:
%load_ext zizai.forms

In [ ]:
%zizai_form

## ⑤ VS CodeにZizAIのスニペットを登録する（任意）

下のセルを1回実行すると、VS Codeの標準ユーザープロファイルで次のスニペットを使えます。

| Pythonセルで入力 | 挿入されるコード | 実行するタイミング |
| --- | --- | --- |
| `@zizai_load` | `%load_ext zizai.forms` | カーネルを起動・再起動してから最初の1回 |
| `@zizai_form` | `%zizai_form` | 読み込み後、フォームを開くたび |

フォームは `@zizai_bq`、`@zizai_duck`、`@zizai_csv`、`@zizai_excel` からも開けます。SQLを書く場合は、挿入された `%zizai_bq` または `%zizai_duck` の先頭に `%` を1つ足し、次の行にSQLを書きます。[接続先と出力の指定方法](../docs/usage.md#connector-magics)

入力して候補を選び、コードセルを実行します。候補が見えない場合は **Ctrl+Space** を押してください。

In [ ]:
import json
from pathlib import Path
from tempfile import NamedTemporaryFile
import os
import sys

if "root" not in globals():
    raise RuntimeError("先にセル0を実行してください。")

if sys.platform == "win32":
    settings_home = Path(os.environ["APPDATA"])
elif sys.platform == "darwin":
    settings_home = Path.home() / "Library" / "Application Support"
else:
    settings_home = Path(os.environ.get("XDG_CONFIG_HOME", Path.home() / ".config"))

source = root / "examples" / "zizai-notebook.code-snippets"
snippet = source.read_text(encoding="utf-8-sig")
published = json.loads(snippet)
previous_split = {label: published[label] for label in ("ZizAIを読み込む", "ZizAIフォームを開く")}
previous_sql_snippets = {**published, **{
    "BigQueryでSQLを実行する": {
        "scope": "python",
        "prefix": "@zizai_bq_sql",
        "body": [
            "%%zizai_bq",
            "$0"
        ],
        "description": "BigQueryでセル内のSQLを実行する"
    },
    "DuckDBでSQLを実行する": {
        "scope": "python",
        "prefix": "@zizai_duck_sql",
        "body": [
            "%%zizai_duck",
            "$0"
        ],
        "description": "DuckDBでセル内のSQLを実行する"
    }
}}
legacy = {
    "ZizAIフォームを開く": {
        "scope": "python",
        "prefix": "@zizai_form",
        "body": [
            "%load_ext zizai.forms",
            "%form"
        ],
        "description": "ZizAIフォームを開く"
    }
}
user_folders = [settings_home / edition / "User" for edition in ("Code", "Code - Insiders")]
user_folders = [folder for folder in user_folders if folder.is_dir()]
if not user_folders:
    raise FileNotFoundError(f"VS Codeのユーザー設定フォルダーを確認してください: {settings_home}")

for user_folder in user_folders:
    folder = user_folder / "snippets"
    folder.mkdir(exist_ok=True)
    destination = folder / source.name
    try:
        with destination.open("x", encoding="utf-8", newline="\n") as file:
            file.write(snippet)
    except FileExistsError:
        existing = destination.read_text(encoding="utf-8-sig")
        if existing != snippet:
            try:
                previous = json.loads(existing)
            except json.JSONDecodeError:
                previous = None
            if previous not in (legacy, previous_split, previous_sql_snippets):
                print(f"別の内容が登録済みです。既存のファイルを保持しました: {destination}")
                continue
            with NamedTemporaryFile(dir=folder, mode="w", encoding="utf-8", newline="\n", delete=False) as staged:
                staged.write(snippet)
            pending = Path(staged.name)
            try:
                pending.replace(destination)
            finally:
                pending.unlink(missing_ok=True)
    print(f"ZizAIのスニペットを利用できます: {destination}")

## Python・Kiroから直接使う場合

同じPython環境で、登録したコネクタを通常のPython importで使えます。たとえば、CSVを選んだ場合は `import csv_connector` です。マジックの読み込みは、Notebookでフォームを表示する場合に必要です。[導入手順](../docs/usage.md)・[Kiro向けガイド](../.kiro/steering/zizai.md)。